## Project: Fake/Real News Detection

In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import re
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [16]:
#load dataset

df = pd.read_csv(r"C:\Users\Naim Prottay\OneDrive\Desktop\Fake_Or_Real_news\dataset\news_data.csv", low_memory=False)

In [17]:
df.head()

,text,label,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,Unnamed: 135,Unnamed: 136,Unnamed: 137,Unnamed: 138
0,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,U.S. Secretary of State John F. Kerry said Mon...,REAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,It's primary day in New York and front-runners...,REAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df.shape

(7795, 139)

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7795 entries, 0 to 7794
Columns: 139 entries, text to Unnamed: 138
dtypes: object(139)
memory usage: 8.3+ MB


In [20]:
df.describe()

,text,label,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,Unnamed: 135,Unnamed: 136,Unnamed: 137,Unnamed: 138
count,6929,6755,318,241,179,133,91,65,55,38,...,1,1,1,1,1,1,1,1,1,1
unique,6644,437,315,241,179,133,91,65,55,38,...,1,1,1,1,1,1,1,1,1,1
top,"Killing Obama administration rules, dismantlin...",REAL,etc.,bold way,” Maureen Dowd writes. “Before he jumped into ...,Trump was seen as bombastic,vulgar,a bit of a buffoon and a cave man,but [also] ‘a cheeky brio.’ He was not regard...,not a predator … But he created another chara...,...,rather than acting out of a sober evaluation ...,but to me,more and more,the whole shooting match seems to come down t...,if one could discredit the idea of American e...,then it becomes much easier to have a serious...,a presidential candidate,a senator can stand behind the podium and mak...,it just seems that undertaking a serious crit...,REAL
freq,58,3161,3,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [21]:
df = df[['text','label']]

In [22]:
df.head(3)

,text,label
0,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,U.S. Secretary of State John F. Kerry said Mon...,REAL


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7795 entries, 0 to 7794
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    6929 non-null   object
 1   label   6755 non-null   object
dtypes: object(2)
memory usage: 121.9+ KB


#### The dataset was loaded and only the relevent columns were retained for furthur analysis.

In [24]:
df['label'].value_counts()

label
REAL                                                                                                                                                                                                           3161
FAKE                                                                                                                                                                                                           3154
 or naturalization or by jus sanguinis – inherited through ancestors/parents                                                                                                                                      3
 etc.                                                                                                                                                                                                             2
 Philippines                                                                                                                                      

In [25]:
valid_labels = ['REAL', 'FAKE']

In [26]:
df = df[df['label'].isin(valid_labels)]

In [27]:
df['label'].value_counts()

label
REAL    3161
FAKE    3154
Name: count, dtype: int64

#### Only valid target labels were retained to ensure correct supervised learning.

## Baseline Sanity Check
#### A baseline approach was use first to assess whether the dataset contains sufficent signal for classification before applying advanced processing techniques.
#### Count Vectorizer will be used for word processing.

In [28]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [29]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words='english')


In [30]:
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [31]:
mnb = MultinomialNB()

In [32]:
y_pred = mnb.fit(X_train_vec, y_train).predict(X_test_vec)

In [33]:
accuracy = accuracy_score(y_test, y_pred)
cls_report = classification_report(y_test, y_pred)
cnf_mat = confusion_matrix(y_test, y_pred)

In [34]:
print(f"Accuracy: {accuracy}")
print(f"Classification Report: {cls_report}")
print(f"confusion Matrix: {cnf_mat}")

Accuracy: 0.9034045922406968
Classification Report:               precision    recall  f1-score   support

        FAKE       0.92      0.88      0.90       631
        REAL       0.89      0.93      0.91       632

    accuracy                           0.90      1263
   macro avg       0.90      0.90      0.90      1263
weighted avg       0.90      0.90      0.90      1263

confusion Matrix: [[555  76]
 [ 46 586]]


## Baseline Sanity Analysis
#### With raw text this kind of result is quite impressive. Accuracy is 90.3%, Recall is 93%. Overall strong model performance. But Count Vectorizer is not suitable for this kind of sensitive model though our model performance is quite good enough. Because Count Vectorizer can't understand the importance of words.

## Natural Language Processing--> Manual

In [35]:
def lower_text(text):
    text = text.lower()
    return text

def remove_punc(text):
    text = re.sub(r"[^\w\s!?]", "", text)
    return text

def remove_whitespace(text):
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [36]:
def normalize_text(text):
    if pd.isna(text):
        return text
    
    text = lower_text(text)
    text = remove_punc(text)
    text = remove_whitespace(text)
    return text

In [37]:
df['text'] = df['text'].apply(normalize_text)

In [38]:
df['text'].head(3)

0    daniel greenfield a shillman journalism fellow...
1    google pinterest digg linkedin reddit stumbleu...
2    us secretary of state john f kerry said monday...
Name: text, dtype: object

In [39]:
X = df['text']
y = df['label']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y, test_size=0.2, random_state=42)

In [40]:
cl_vectorizer = CountVectorizer(stop_words='english')

In [41]:

X_train_vec_c = cl_vectorizer.fit_transform(X_train_c)
X_test_vec_c = cl_vectorizer.transform(X_test_c)

In [42]:
mnb_clean = MultinomialNB()

y_pred_c = mnb_clean.fit(X_train_vec_c, y_train_c).predict(X_test_vec_c)

In [43]:
accuracy_c = accuracy_score(y_test_c, y_pred_c)
cls_report_c = classification_report(y_test_c, y_pred_c)
cnf_mat_c = confusion_matrix(y_test_c, y_pred_c)

In [44]:
print(f"Accuracy(Clean): {accuracy_c}")
print(f"Classification Report: {cls_report_c}")
print(f"Confusion Matrix: {cnf_mat_c}")

Accuracy(Clean): 0.8907363420427553
Classification Report:               precision    recall  f1-score   support

        FAKE       0.92      0.85      0.88       616
        REAL       0.87      0.93      0.90       647

    accuracy                           0.89      1263
   macro avg       0.89      0.89      0.89      1263
weighted avg       0.89      0.89      0.89      1263

Confusion Matrix: [[525  91]
 [ 47 600]]


## NLP Model Analysis
#### After the NLP pipeline, new model is also close to baseline model. But we overclean the text's, thats why model can't catch full meaning of a word and why it is used. After NLP pipeline implementions, we can assume why Count Vectorizer is not good with clean sensitive texts, what we talked earlier. So lets try TF-IDF and then with Bigram.

In [45]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_df=0.9,
    min_df=5
)

X_train_tf = tfidf.fit_transform(X_train)
X_test_tf = tfidf.transform(X_test)

mnb_tf = MultinomialNB()

y_pred_tf = mnb_tf.fit(X_train_tf, y_train).predict(X_test_tf)

In [46]:
accuracy_tf = accuracy_score(y_test, y_pred_tf)
cl_report_tf = classification_report(y_test, y_pred_tf)
cnf_matrix_tf = confusion_matrix(y_test, y_pred_tf)

In [47]:
print(f"Accuracy Score: {accuracy_tf}")
print(f"classification_report: {cl_report_tf}")
print(f"Confusion Matrix: {cnf_matrix_tf}")

Accuracy Score: 0.8962787015043547
classification_report:               precision    recall  f1-score   support

        FAKE       0.93      0.85      0.89       631
        REAL       0.87      0.94      0.90       632

    accuracy                           0.90      1263
   macro avg       0.90      0.90      0.90      1263
weighted avg       0.90      0.90      0.90      1263

Confusion Matrix: [[539  92]
 [ 39 593]]


## TF-IDF Analysis
#### So we've finished TF-IDF, But not that much improvement. Accuracy is like as previous model. Recall has raised, but increase fake news mis(92). Lets try with Bigram.

## N-Grams

In [48]:
#tf-idf
tfidf_ngram = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9
)

## Implementing Multinomial Naive Bayes Model

In [49]:
X_train_nrg = tfidf_ngram.fit_transform(X_train)
X_test_nrg = tfidf_ngram.transform(X_test)

mnb_nrg = MultinomialNB()
y_pred_nrg = mnb_nrg.fit(X_train_nrg, y_train).predict(X_test_nrg)

In [50]:
acc = accuracy_score(y_test, y_pred_nrg)
cl_rep = classification_report(y_test, y_pred_nrg)
cnf_mat = confusion_matrix(y_test, y_pred_nrg)

In [51]:
print(f"Accuracy: {acc}")
print(f"classification_report: {cl_rep}")
print(f"Confusion Matrix: {cnf_mat}")

Accuracy: 0.9081551860649247
classification_report:               precision    recall  f1-score   support

        FAKE       0.95      0.86      0.90       631
        REAL       0.88      0.95      0.91       632

    accuracy                           0.91      1263
   macro avg       0.91      0.91      0.91      1263
weighted avg       0.91      0.91      0.91      1263

Confusion Matrix: [[545  86]
 [ 30 602]]


## TF-IDF+Bigram Analysis
#### Result is quite good now. Accuracy meets 90.8%. Other metrics is also good. Model miss 86 fake news, but real news detection is improved. 
#### So we can say this model is production friendly.

### Limitations & Future Improvements:
* Implement Logistic Regression / Linear SVM
* Hyperparameter tuning with GridSearchCV
* Lemmatization instead of aggressive cleaning
* Add ROC-AUC for threshold analysis
